In [3]:
import re
from pathlib import Path

import pandas as pd

In [4]:
# version = "eval_4"
version = "eval_5"
paths = sorted(Path("output/input_space_v2").rglob(f"*/{version}/*/eval_table.csv"))
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, trial = match.groups()
    table.insert(0, "trial", int(trial))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
table = pd.concat(tables, ignore_index=True)
print(table.shape)
table.head()

47
(164, 19)


,key,space,base_lr,trial,model,repr,clf,dataset,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,13,0.00270,0.05,30,"[2.7, 1.0]",train,0.000459,0.999947,0.000052,0.999943,0.000056
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,13,0.00270,0.05,30,"[2.7, 1.0]",validation,0.087002,0.988591,0.001661,0.987383,0.002114
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,13,0.00270,0.05,30,"[2.7, 1.0]",test,0.109485,0.985714,0.001607,0.982369,0.002132
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,6,0.00023,0.05,15,"[0.23, 1.0]",train,2.126593,0.362089,0.002424,0.306910,0.002518
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,6,0.00023,0.05,15,"[0.23, 1.0]",validation,2.397593,0.283315,0.005435,0.225098,0.005092


In [5]:
summary = table.loc[table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "trial", "repr", "clf"], columns="dataset"
)
summary = summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
summary

acc                   acc_std             
dataset                   hcpya_task21 nsd_cococlip hcpya_task21 nsd_cococlip
space       base_lr trial                                                    
flat        0.0010  1         0.985714     0.301484     0.001607     0.005486
                    2         0.985119     0.291466     0.001627     0.005472
                    3         0.989484     0.306494     0.001421     0.005333
                    4         0.985913     0.295733     0.001557     0.005651
                    5         0.986310     0.281633     0.001569     0.005160
                    6         0.985913     0.308905     0.001633     0.005616
                    7         0.986508     0.292393     0.001608     0.005301
                    8         0.984524     0.315028     0.001673     0.005465
mni_cortex  0.0010  1         0.957738     0.268831     0.002949     0.005189
                    2         0.951984     0.259369     0.003115     0.004898
                    3         0.962302     0.263636     0.002433     0.004876
                    4         0.959921     0.258627     0.002841     0.005060
                    5         0.959722     0.270130     0.002792     0.005177
                    6         0.955952          NaN     0.002810          NaN
                    7         0.961111     0.259184     0.002679     0.005042
                    8         0.951389     0.268089     0.003084     0.005041
schaefer400 0.0003  1         0.975794     0.266790     0.002218     0.004823
                    2         0.974405     0.259369     0.002309     0.004973
                    3         0.971825     0.269017     0.002270     0.005349
                    4         0.977778     0.275696     0.002192     0.005533
                    5         0.977579     0.287384     0.002066     0.005566
                    6         0.977778     0.277737     0.002065     0.005131
                    7         0.970635     0.267532     0.002462     0.005172
                    8         0.976587     0.274768     0.002168     0.005406

In [6]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [7]:
# datasets = ["hcpya_rest1lr_gender", "aabc_sex", "hcpya_task21", "nsd_cococlip"]
# datasets = ["aabc_age", "hcpya_rest1lr_gender", "hcpya_task21", "nsd_cococlip"]
datasets = ["hcpya_task21", "nsd_cococlip"]

space_lrs = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

records = []

for (space, base_lr, trial), row in summary.iterrows():
    if base_lr != space_lrs[space]:
        continue
    record = {"space": space, "lr": base_lr, "trial": trial}
    for ds in datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   trial | HCP-YA Task21   | NSD COCO24   |
|:------------|-------:|--------:|:----------------|:-------------|
| flat        | 0.001  |       1 | 98.6 ± 0.2      | 30.1 ± 0.5   |
| flat        | 0.001  |       2 | 98.5 ± 0.2      | 29.1 ± 0.5   |
| flat        | 0.001  |       3 | 98.9 ± 0.1      | 30.6 ± 0.5   |
| flat        | 0.001  |       4 | 98.6 ± 0.2      | 29.6 ± 0.6   |
| flat        | 0.001  |       5 | 98.6 ± 0.2      | 28.2 ± 0.5   |
| flat        | 0.001  |       6 | 98.6 ± 0.2      | 30.9 ± 0.6   |
| flat        | 0.001  |       7 | 98.7 ± 0.2      | 29.2 ± 0.5   |
| flat        | 0.001  |       8 | 98.5 ± 0.2      | 31.5 ± 0.5   |
| mni_cortex  | 0.001  |       1 | 95.8 ± 0.3      | 26.9 ± 0.5   |
| mni_cortex  | 0.001  |       2 | 95.2 ± 0.3      | 25.9 ± 0.5   |
| mni_cortex  | 0.001  |       3 | 96.2 ± 0.2      | 26.4 ± 0.5   |
| mni_cortex  | 0.001  |       4 | 96.0 ± 0.3      | 25.9 ± 0.5   |
| mni_cortex  | 0.001  |       5 | 96.0 ± 0.3   

In [6]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


logistic_summary = table.loc[(table["split"] == "test") & (table["clf"] == "logistic")].pivot_table(
    values=["acc"],
    index=["space", "base_lr", "trial", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
logistic_summary = logistic_summary.loc[
    (slice(None), slice(None), slice(None), "cls", "logistic")
].dropna(axis=1, how="all")
logistic_summary

mean                                 \
                                acc                                  
dataset                    aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr trial                                            
flat        0.0010  1      0.408333  0.876705  0.592944   0.609690   
                    2      0.447143  0.870057  0.576089   0.580000   
                    3      0.421190  0.868580  0.608065   0.599302   
                    4      0.436488  0.874602  0.568065   0.608527   
                    5      0.439286  0.873239  0.572702   0.607907   
mni_cortex  0.0010  1      0.522202  0.855000  0.593790   0.603488   
                    2      0.535417  0.852273  0.576129   0.593566   
                    3      0.524286  0.864432  0.587339   0.595969   
                    4      0.531190  0.856591  0.587056   0.601705   
                    5      0.516905  0.865341  0.591129   0.616977   
schaefer400 0.0003  1      0.433036  0.711250  0.609194   0.608682   
                    2      0.438333  0.691080  0.617460   0.590698   
                    3      0.425774  0.691193  0.604879   0.582946   
                    4      0.440833  0.709205  0.632016   0.593023   
                    5      0.433274  0.697898  0.602661   0.585194   

                                                                        \
                                                                         
dataset                   adni_ad_vs_cn hcpya_rest1lr_gender   ppmi_dx   
space       base_lr trial                                                
flat        0.0010  1          0.767561             0.869333  0.634271   
                    2          0.758293             0.877056  0.621658   
                    3          0.760407             0.846778  0.639950   
                    4          0.767154             0.878667  0.627035   
                    5          0.760407             0.868278  0.629598   
mni_cortex  0.0010  1          0.771951             0.908889  0.630000   
                    2          0.769675             0.901111  0.629698   
                    3          0.756016             0.899556  0.639497   
                    4          0.775854             0.919778  0.631256   
                    5          0.761463             0.892778  0.624121   
schaefer400 0.0003  1          0.762764             0.800000  0.629648   
                    2          0.762276             0.794333  0.642161   
                    3          0.759675             0.800500  0.634724   
                    4          0.759431             0.813611  0.643216   
                    5          0.757886             0.789556  0.646482   

                                sem                                 \
                                acc                                  
dataset                    aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr trial                                            
flat        0.0010  1      0.002815  0.002483  0.002803   0.003701   
                    2      0.003347  0.002581  0.002708   0.003690   
                    3      0.002989  0.002181  0.002534   0.003588   
                    4      0.003319  0.002408  0.002832   0.003973   
                    5      0.003384  0.002258  0.002735   0.003994   
mni_cortex  0.0010  1      0.003397  0.002482  0.002529   0.003611   
                    2      0.003296  0.002469  0.002622   0.003633   
                    3      0.003322  0.002586  0.002477   0.003478   
                    4      0.002781  0.002467  0.002867   0.003449   
                    5      0.003200  0.002441  0.002706   0.003980   
schaefer400 0.0003  1      0.003477  0.003389  0.002366   0.003429   
                    2      0.002789  0.002970  0.002814   0.003562   
                    3      0.003070  0.002912  0.002594   0.003296   
                    4      0.003202  0.003396  0.002739   0.003923   
                    5      0.0030

In [11]:
# datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex", "hcpya_rest1lr_gender"]
datasets = ["adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex", "hcpya_rest1lr_gender"]

space_lrs = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

records = []

for (space, base_lr, trial), row in logistic_summary.iterrows():
    if base_lr != space_lrs[space]:
        continue
    record = {"space": space, "lr": base_lr, "trial": trial}
    for ds in datasets:
        acc = row[("mean", "acc", ds)]
        std = row[("sem", "acc", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   trial | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   | HCP-YA Sex   |
|:------------|-------:|--------:|:-----------|:-----------|:------------|:------------|:-------------|
| flat        | 0.001  |       1 | 76.8 ± 0.3 | 63.4 ± 0.3 | 40.8 ± 0.3  | 87.7 ± 0.2  | 86.9 ± 0.2   |
| flat        | 0.001  |       2 | 75.8 ± 0.2 | 62.2 ± 0.2 | 44.7 ± 0.3  | 87.0 ± 0.3  | 87.7 ± 0.2   |
| flat        | 0.001  |       3 | 76.0 ± 0.2 | 64.0 ± 0.2 | 42.1 ± 0.3  | 86.9 ± 0.2  | 84.7 ± 0.3   |
| flat        | 0.001  |       4 | 76.7 ± 0.2 | 62.7 ± 0.3 | 43.6 ± 0.3  | 87.5 ± 0.2  | 87.9 ± 0.2   |
| flat        | 0.001  |       5 | 76.0 ± 0.2 | 63.0 ± 0.2 | 43.9 ± 0.3  | 87.3 ± 0.2  | 86.8 ± 0.2   |
| mni_cortex  | 0.001  |       1 | 77.2 ± 0.2 | 63.0 ± 0.2 | 52.2 ± 0.3  | 85.5 ± 0.2  | 90.9 ± 0.2   |
| mni_cortex  | 0.001  |       2 | 77.0 ± 0.2 | 63.0 ± 0.2 | 53.5 ± 0.3  | 85.2 ± 0.2  | 90.1 ± 0.2   |
| mni_cortex  | 0.001  |       3 | 75.6 ± 0.2 | 63.9 ± 0.2 | 52.